# **Image Inpainting** with **LangSAM** and **Stable Diffusion:**

### The **lang-segment-anything** library presents an innovative approach to object detection and segmentation by combining the strengths of **GroundingDino** and **SAM**. SAM’s default image encoder is **ViT-H**, but it can also utilize ViT-L or ViT-B depending on the specific requirements


In [ ]:
!pip install -U git+https://github.com/luca-medeiros/lang-segment-anything.git

## **Diffusers** library for **Inpainting** the segmented are based on Target-Prompt.

In [ ]:
!pip install diffusers transformers accelerate scipy safetensors

### Dependencies

In [ ]:
from PIL import Image
from lang_sam import LangSAM
from lang_sam.utils import draw_image
from diffusers import StableDiffusionInpaintPipeline
import matplotlib.pyplot as plt
import torch
import numpy as np

In [ ]:
# Check and set device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

### Initialize **LangSAM** model for object segmentation

In [ ]:
model = LangSAM()

### Initialize **StableDiffusionInpaintPipeline** model for image inpainting

In [ ]:
inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-inpainting",
    torch_dtype=torch.float16)

inpaint.to(device)

### End-to-end Pipeline


In [ ]:
# Function to process image, perform segmentation, and inpainting

def final_pipeline(image_pil, Source_prompt, Target_prompt):

    # Step 1: LangSAM predict
    results = model.predict(image_pil, Source_prompt)
    result = results[0]

    masks = result["masks"]
    boxes = result["boxes"]
    labels = result["labels"]
    logits = result["scores"]
    mask_scores = result['mask_scores']

    # Step 2: Select best mask based on LangSAM masking
    best_idx = np.argmax(mask_scores)
    mask = masks[best_idx]

    # Step 3: Convert mask to PIL Image
    mask = (mask > 0.5).astype(np.uint8) * 255
    mask_pil = Image.fromarray(mask)

    # Step 4: Perform inpainting with StableDiffusionInpaintPipeline
    inpainted_image = inpaint(prompt=Target_prompt, image=image_pil, mask_image=mask_pil).images[0]

    # Step 5: Putting mask on Image for showing Comparison
    overlay = np.array(image_pil).copy()
    if overlay.ndim == 4:
      overlay = overlay[0]
    overlay[mask == 255] = [0, 0, 0]
    overlay_pil = Image.fromarray(overlay)

    return inpainted_image, overlay_pil

### Function to plot Original image, Mask, Inpainted image

In [ ]:
def plot(image_pil, inpainted_image, mask):
  fig, axes = plt.subplots(1, 3, figsize=(12, 6))

  # Plot original image with masks
  axes[0].imshow(image_pil)
  axes[0].set_title('Original Image')

  # Plot inpainted image
  axes[2].imshow(inpainted_image)
  axes[2].set_title('Inpainted Image')

  # Plot Mask
  axes[1].imshow(mask)
  axes[1].set_title('Mask')

  # Hide axes ticks for cleaner visualization
  for ax in axes:
      ax.axis('off')

  # Adjust layout and display images
  plt.tight_layout()
  plt.show()


### EXAMPLE 1


In [ ]:
image_pil = Image.open('/content/Screenshot 2024-06-25 114812.png').convert("RGB").resize((512, 512))
Source_prompt = 'House'
Target_prompt = "middle-age Castle"
inpainted_image, mask = final_pipeline([image_pil], [Source_prompt], Target_prompt)

plot(image_pil, inpainted_image, mask)

### EXAMPLE 2



In [ ]:
image_pil = Image.open('/content/Screenshot 2026-05-04 124403.png').convert("RGB").resize((512, 512))
Source_prompt = 'Car'
Target_prompt = "Race Car"
inpainted_image, mask = final_pipeline([image_pil], [Source_prompt], Target_prompt)

plot(image_pil, inpainted_image, mask)